# Shelf Wave Profiles and the Compacton Singular Limit
## Lax Centennial Paper: Section 6 — Theorem 6.3 Figures

This notebook generates the shelf wave profile data and pgfplots CSV
files used in Figure 3 (pgfplots-generated) of the main paper.

As background a → 0, smooth shelf waves with exponential tails
converge to compactons with compact support — the singular limit
established in Theorem 6.3.

**Reference:** J.M. Hyman, "What Remains When Integrability Fails?",
DCDS-A Lax Centennial Volume (2026), Section 6.

## ① Setup

In [ ]:
import numpy as np
if not hasattr(np, 'trapz'): np.trapz = np.trapezoid  # NumPy 2 renamed trapz
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import csv, os

output_dir = 'figdata'
os.makedirs(output_dir, exist_ok=True)

## ② Compacton Profile (a = 0 limit)

In [ ]:
def compacton_k22(x, A=1.0, x0=0.0):
    """K(2,2) compacton: u = A cos^2(xi/4) for |xi| < 2*pi."""
    xi = x - x0
    u = np.zeros_like(x)
    inside = np.abs(xi) < 2*np.pi
    u[inside] = A * np.cos(xi[inside] / 4.0)**2
    return u

# Generate compacton profile CSV
x_plot = np.linspace(-10, 10, 1001)
u_comp = compacton_k22(x_plot)

with open(f'{output_dir}/profile_compacton.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['x', 'u'])
    for xi, ui in zip(x_plot, u_comp):
        w.writerow([f'{xi:.6f}', f'{ui:.10f}'])
print("Saved profile_compacton.csv")

## ③ Shelf Wave ODE Solver

In [ ]:
def shelf_wave_k22(a, A_peak=1.0, xi_max=10.0, N=2001):
    """Solve K(2,2) traveling-wave ODE on background u = a.
    
    ODE: 2U U'' + 2(U')^2 - cU + U^2 = C
    where C = -c*a + a^2, c chosen so peak amplitude = A_peak.
    """
    c = 3.0/4.0 * A_peak**(1)  # K(2,2) speed for peak A
    # Adjust c for background: c = (3/4)(A_peak^2 - a^2)/(A_peak - a) approx
    # For simplicity use shooting from the peak
    C_const = -c * a + a**2
    
    def ode(xi, y):
        U, Up = y
        if abs(U) < 1e-14:
            return [Up, 0.0]
        Upp = (C_const + c*U - U**2 - 2*Up**2) / (2*U)
        return [Up, Upp]
    
    # Integrate from peak (U=A_peak, U'=0) outward
    xi_fwd = np.linspace(0, xi_max, N//2 + 1)
    xi_bwd = np.linspace(0, -xi_max, N//2 + 1)
    
    sol_f = solve_ivp(ode, [0, xi_max], [A_peak, 0.0],
                      t_eval=xi_fwd, rtol=1e-10, atol=1e-12)
    sol_b = solve_ivp(ode, [0, -xi_max], [A_peak, 0.0],
                      t_eval=xi_bwd, rtol=1e-10, atol=1e-12)
    
    xi_full = np.concatenate([sol_b.t[::-1][:-1], sol_f.t])
    U_full = np.concatenate([sol_b.y[0,::-1][:-1], sol_f.y[0,:]])
    return xi_full, U_full

## ④ Generate Profile Family

In [ ]:
a_values = [0.01, 0.10, 0.50, 1.00]
profiles = {}

for a in a_values:
    xi, U = shelf_wave_k22(a, A_peak=1.0, xi_max=10.0)
    profiles[a] = (xi, U)
    
    fname = f'{output_dir}/profile_a{a:.2f}.csv'
    with open(fname, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['x', 'u'])
        for xv, uv in zip(xi, U):
            w.writerow([f'{xv:.6f}', f'{uv:.10f}'])
    print(f"Saved {fname}")

## ⑤ Decay Rate Validation (Table S5)

In [ ]:
print(f"{'a':>8} {'kappa_theory':>14} {'kappa_meas':>14} {'rel_err':>10}")
print("-" * 50)

for a in a_values:
    xi, U = profiles[a]
    # Theoretical decay rate
    c = 3.0/4.0  # speed for unit amplitude K(2,2)
    c_star = 2.0 * a  # m*a^{m-1} for m=2
    if c > c_star:
        kappa_thy = np.sqrt((c - c_star) / c_star)
    else:
        kappa_thy = np.nan
    
    # Measured: fit exponential to right tail
    right = xi > 5.0
    if np.any(right) and not np.isnan(kappa_thy):
        U_tail = U[right] - a  # subtract background
        xi_tail = xi[right]
        pos = U_tail > 1e-8
        if np.sum(pos) > 5:
            log_U = np.log(U_tail[pos])
            coeffs = np.polyfit(xi_tail[pos], log_U, 1)
            kappa_meas = -coeffs[0]
            rel_err = abs(kappa_meas - kappa_thy) / kappa_thy
            print(f"{a:8.3f} {kappa_thy:14.6f} {kappa_meas:14.6f} {rel_err:10.4%}")
        else:
            print(f"{a:8.3f} {kappa_thy:14.6f} {'N/A':>14} {'N/A':>10}")
    else:
        print(f"{a:8.3f} {'N/A':>14} {'N/A':>14}")

## ⑥ Publication Figure

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: Profile family
ax1.plot(x_plot, u_comp, 'k-', linewidth=2, label='Compacton ($a=0$)')
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for (a, color) in zip(a_values, colors):
    xi, U = profiles[a]
    ax1.plot(xi, U, color=color, linewidth=1.2, label=f'$a = {a}$')
ax1.set_xlabel(r'$\xi = x - ct$')
ax1.set_ylabel(r'$U(\xi)$')
ax1.set_title('Shelf wave profiles converging to compacton')
ax1.legend(frameon=False)
ax1.set_xlim(-10, 10)

# Right: Log-scale tail decay
for (a, color) in zip(a_values, colors):
    xi, U = profiles[a]
    right = xi > 1.0
    tail = U[right] - a
    pos = tail > 1e-10
    if np.any(pos):
        ax2.semilogy(xi[right][pos], tail[pos], color=color, label=f'$a={a}$')
ax2.set_xlabel(r'$\xi$')
ax2.set_ylabel(r'$U(\xi) - a$')
ax2.set_title('Exponential tail decay (log scale)')
ax2.legend(frameon=False)

plt.tight_layout()
plt.savefig('shelf_wave_profiles.pdf', dpi=300)
print("Saved shelf_wave_profiles.pdf")
plt.show()

## ⑦ Period–Energy Data

In [ ]:
# Generate period-energy relationship data for pgfplots
# For K(2,2): T(E) relationship via numerical ODE solution
a_range = np.logspace(-3, 0, 50)
periods = []
energies = []

for a in a_range:
    xi, U = shelf_wave_k22(a, A_peak=1.0, xi_max=30.0, N=4001)
    # Period: estimate from profile width at half-maximum
    above_half = U > (U.max() + a) / 2.0
    if np.any(above_half):
        idx = np.where(above_half)[0]
        width = xi[idx[-1]] - xi[idx[0]]
    else:
        width = np.nan
    # Energy: integral of (U-a)^2
    E = np.trapz((U - a)**2, xi)
    periods.append(width)
    energies.append(E)

with open(f'{output_dir}/period_energy.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['a', 'period', 'energy'])
    for a, T, E in zip(a_range, periods, energies):
        w.writerow([f'{a:.6e}', f'{T:.6f}', f'{E:.10e}'])
print(f"Saved period_energy.csv ({len(a_range)} points)")